# Notebook 4 — ML Models

**Goal:** Train and understand XGBoost — the gradient-boosted tree model that is the workhorse of structured/tabular machine learning.

## Why gradient-boosted trees?

Linear models (Ridge) assume the relationship between features and outcome is *additive and linear*. But train delays aren't linear:
- A train being 5 minutes late **AND** it being peak hour is worse than either alone
- The `is_delayed` flag matters a lot more at long horizons than short ones
- ETA drift matters only if the train is at least 3 minutes away

These are **interactions** and **nonlinear thresholds** — decision trees model them naturally.

## How gradient boosting works (intuition)

1. Fit a simple decision tree to the data. It makes some errors.
2. Fit a second tree to the *errors* of the first tree (the residuals).
3. Add the two trees together. The combined model is more accurate.
4. Repeat 300–500 times, each tree correcting the mistakes of all previous trees.
5. The final model is an **ensemble** — a weighted sum of hundreds of small trees.

"Gradient" because each step minimises the loss function by following its gradient — same idea as gradient descent in neural networks, but applied to building trees.

## Key hyperparameters

| Parameter | What it controls | Typical range |
|-----------|-----------------|---------------|
| `n_estimators` | Number of trees | 100–1000 |
| `max_depth` | Max depth of each tree (higher = more complex, more overfit risk) | 3–8 |
| `learning_rate` | How much each tree contributes (smaller = more trees needed, but often better) | 0.01–0.3 |
| `subsample` | Fraction of rows sampled per tree (adds randomness, reduces overfit) | 0.6–1.0 |
| `colsample_bytree` | Fraction of features sampled per tree | 0.6–1.0 |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    accuracy_score, f1_score
)
import xgboost as xgb

sns.set_theme(style='darkgrid')

df = pd.read_parquet('../data/features.parquet')
df = df.sort_values('snapshot_time')

cutoff = df['snapshot_time'].max() - pd.Timedelta(days=14)
train = df[df['snapshot_time'] <= cutoff].copy()
test  = df[df['snapshot_time'] >  cutoff].copy()
print(f'Train: {len(train):,}  Test: {len(test):,}')

FEATURE_COLS = [
    'is_red', 'is_blue', 'stop_sequence', 'direction',
    'hour', 'dow', 'is_weekend', 'is_peak_am', 'is_peak_pm',
    'minutes_until', 'is_scheduled', 'is_delayed', 'is_faulty',
    'eta_delta_1', 'eta_delta_2',
]

def add_status(df):
    def label(d):
        if d < -1: return 'ahead'
        if d <= 2: return 'on_time'
        return 'behind'
    df = df.copy()
    df['status'] = df['delay_minutes'].apply(label)
    return df

train = add_status(train)
test  = add_status(test)

X_train = train[FEATURE_COLS].fillna(0)
X_test  = test[FEATURE_COLS].fillna(0)
y_train = train['delay_minutes']
y_test  = test['delay_minutes']

LABELS = ['ahead', 'on_time', 'behind']
label_map = {l: i for i, l in enumerate(LABELS)}
y_train_cls = train['status'].map(label_map)
y_test_cls  = test['status'].map(label_map)

## Model 1: XGBoost Regressor

We first train a regression model — it directly predicts `delay_minutes` as a number. We can then threshold the number to get the status label.

**Early stopping:** We pass a validation set and stop adding trees once the validation loss stops improving. This prevents overfitting and automatically finds the right `n_estimators`.

In [ ]:
reg = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    early_stopping_rounds=30,
    eval_metric='mae',
)

reg.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)

print(f'Best iteration: {reg.best_iteration} (early stopped from 500)')

pred_reg = reg.predict(X_test)
mae  = mean_absolute_error(y_test, pred_reg)
rmse = mean_squared_error(y_test, pred_reg, squared=False)
print(f'XGBoost Regressor  MAE={mae:.3f}  RMSE={rmse:.3f}')

In [ ]:
# Learning curves — watching train vs validation MAE as we add trees
# This is essential for diagnosing overfitting.
#
# If train loss keeps dropping but val loss stops improving → overfitting
# If both keep dropping together → underfitting (need more trees)

evals = reg.evals_result()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(evals['validation_0']['mae'], label='Train MAE', alpha=0.7)
ax.plot(evals['validation_1']['mae'], label='Test MAE', alpha=0.9)
ax.axvline(reg.best_iteration, color='orange', linestyle='--', label=f'Best iteration ({reg.best_iteration})')
ax.set_title('XGBoost learning curves')
ax.set_xlabel('Number of trees')
ax.set_ylabel('MAE (minutes)')
ax.legend()
plt.tight_layout()
plt.show()

## Feature importance

XGBoost can tell us which features it used most heavily. There are several ways to measure this:

- **`weight`**: how many times a feature was split on (counts)
- **`gain`**: average improvement in loss from splits using this feature ← most useful
- **`cover`**: how many rows were affected by splits on this feature

Feature importance is useful for:
1. **Sanity check** — do the top features match your intuition?
2. **Feature selection** — if a feature has near-zero importance, consider dropping it
3. **Understanding the model** — tells you what the model "thinks" matters

In [ ]:
importances = pd.Series(
    reg.get_booster().get_score(importance_type='gain'),
).rename(index=dict(zip([f'f{i}' for i in range(len(FEATURE_COLS))], FEATURE_COLS)))

# If the above renaming doesn't align, use this:
imp_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': reg.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
imp_df.plot(x='feature', y='importance', kind='barh', ax=ax, legend=False, color='steelblue')
ax.set_title('XGBoost feature importance (gain)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## Model 2: Quantile Regression — measuring uncertainty

Standard regression gives a single number: "I predict this train will be 2.3 minutes late."

But predictions have uncertainty. A better answer might be: "I predict 2.3 minutes late, and I'm 80% confident the actual delay will be between −0.5 and 8 minutes."

**Quantile regression** lets us predict *any percentile* of the outcome distribution.
- `quantile_alpha=0.1` → predict the 10th percentile (optimistic end)
- `quantile_alpha=0.5` → predict the median
- `quantile_alpha=0.9` → predict the 90th percentile (pessimistic end)

The gap between p10 and p90 is your **prediction interval** — it quantifies how confident the model is.

Trains with high uncertainty (wide interval) are ones where you should say "this could go either way" in the UI.

In [ ]:
quantile_models = {}

for alpha, tag in [(0.1, 'p10'), (0.5, 'p50'), (0.9, 'p90')]:
    m = xgb.XGBRegressor(
        objective='reg:quantileerror',
        quantile_alpha=alpha,
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )
    m.fit(X_train, y_train)
    quantile_models[tag] = m
    print(f'{tag} model trained')

pred_p10 = quantile_models['p10'].predict(X_test)
pred_p50 = quantile_models['p50'].predict(X_test)
pred_p90 = quantile_models['p90'].predict(X_test)

In [ ]:
# Calibration check: the p90 interval should contain the actual value 90% of the time.
# Similarly for p10 (actual should exceed p10 90% of the time).

coverage_p90 = (y_test.values <= pred_p90).mean()
coverage_p10 = (y_test.values >= pred_p10).mean()
interval_width = (pred_p90 - pred_p10).mean()

print(f'Actual values ≤ p90 prediction: {coverage_p90:.1%}  (target: 90%)')
print(f'Actual values ≥ p10 prediction: {coverage_p10:.1%}  (target: 90%)')
print(f'Mean p10→p90 interval width: {interval_width:.2f} minutes')
print()
print('Good calibration: coverage near 90%. Over-coverage = intervals too wide (conservative).')
print('Under-coverage = intervals too narrow (overconfident).')

In [ ]:
# Visualise prediction intervals vs actual delay for 50 test examples
sample_idx = np.random.choice(len(test), 50, replace=False)
sample_idx = np.sort(sample_idx)

fig, ax = plt.subplots(figsize=(14, 5))

# Plot intervals
ax.vlines(
    range(50),
    pred_p10[sample_idx], pred_p90[sample_idx],
    color='steelblue', alpha=0.5, linewidth=2, label='p10–p90 interval'
)
ax.scatter(range(50), pred_p50[sample_idx], color='steelblue', s=20, zorder=5, label='p50 (median pred)')
ax.scatter(range(50), y_test.values[sample_idx], color='tomato', s=20, zorder=6, label='Actual delay')

ax.axhline(0, color='white', linestyle='--', alpha=0.4)
ax.set_title('Quantile predictions vs actual delay (50 test samples)')
ax.set_ylabel('Delay (minutes)')
ax.set_xlabel('Test sample')
ax.legend()
plt.tight_layout()
plt.show()

## Model 3: XGBoost Classifier

Instead of predicting a number and thresholding, we can train XGBoost directly as a **3-class classifier** (ahead / on_time / behind).

**Regression + threshold vs direct classification:**
- Regression + threshold is simpler and lets you adjust thresholds after training
- Direct classification optimises the classification loss, so it may produce better class predictions at the cost of losing delay magnitude
- In practice, for this problem, regression + threshold usually wins because the thresholds are somewhat arbitrary anyway

In [ ]:
clf = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

clf.fit(X_train, y_train_cls, eval_set=[(X_test, y_test_cls)], verbose=False)

pred_cls = clf.predict(X_test)
pred_cls_labels = [LABELS[i] for i in pred_cls]

acc = accuracy_score(y_test_cls, pred_cls)
f1  = f1_score(y_test_cls, pred_cls, average='macro', zero_division=0)
print(f'XGBoost Classifier  Accuracy={acc:.3f}  Macro F1={f1:.3f}')

In [ ]:
# Class probabilities: the model doesn't just output a label; it outputs
# a probability for each class. Using these probabilities directly lets
# you communicate confidence in the UI ("likely on time" vs "probably late").

probs = clf.predict_proba(X_test)
probs_df = pd.DataFrame(probs, columns=[f'P({l})' for l in LABELS])
probs_df['true_label'] = [LABELS[i] for i in y_test_cls.values]
probs_df['pred_label'] = pred_cls_labels

print('Sample predictions with probabilities:')
probs_df.sample(8, random_state=1).to_string(float_format='{:.3f}'.format)

## Saving models for the API

In [ ]:
import joblib
from pathlib import Path

model_dir = Path('../ml_models')
model_dir.mkdir(exist_ok=True)

joblib.dump(reg,                    model_dir / 'xgb_regressor.joblib')
joblib.dump(clf,                    model_dir / 'xgb_classifier.joblib')
joblib.dump(quantile_models['p10'], model_dir / 'xgb_p10.joblib')
joblib.dump(quantile_models['p90'], model_dir / 'xgb_p90.joblib')
joblib.dump(label_map,              model_dir / 'label_map.joblib')

print('Models saved to ml_models/')

## Key takeaways

1. **Gradient boosting captures nonlinear interactions** — combinations of features that Ridge can't model.
2. **Learning curves diagnose overfitting** — if test loss flattens while train loss keeps dropping, you need more regularization (lower `learning_rate`, smaller `max_depth`, more `subsample`).
3. **Feature importance is a sanity check** — if `is_faulty` is your top feature, something's wrong.
4. **Quantile regression quantifies uncertainty** — a 2-minute-wide p10–p90 interval means "I'm pretty sure". A 15-minute interval means "anything could happen".
5. **Classification probabilities are more useful than labels** — P(behind) = 0.72 is more actionable than just saying "behind".

➡️ **Next:** [05_evaluation.ipynb](05_evaluation.ipynb)